# Chapter 1: Vectors Live Somewhere

## Inner products, angles, projections, and their meaning in transformer hidden-state space

Every vector lives in a **vector space** — a structured arena where the notions of *length*, *angle*, and *projection* are defined by an **inner product**. This chapter builds your geometric intuition for these ideas and then applies them to the space that matters most for us: the hidden-state space $\mathbb{R}^p$ of a transformer.

A transformer with $L$ layers and $T$ input tokens produces a grid of hidden-state vectors:

$$H^{(l,t)} \in \mathbb{R}^p, \quad l = 0, 1, \ldots, L, \quad t = 1, \ldots, T.$$

Each $H^{(l,t)}$ is a vector — it has a length, it makes an angle with every other vector, and it can be projected onto any direction. Understanding these geometric properties is the first step toward understanding what a transformer *does* to its representations as information flows through layers.

By the end of this chapter you will be able to:
- Compute inner products and interpret them as alignment measures
- Visualize how token representations align (or diverge) across layers
- Articulate what the inner product captures — and what it misses

In [ ]:
import sys, os

# Ensure the project root is on the path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import ltg_ga

print("Imports OK")
print(f"NumPy {np.__version__}")

## The Inner Product

The **inner product** (or dot product) of two vectors $\mathbf{a}, \mathbf{b} \in \mathbb{R}^k$ is defined as:

$$\mathbf{a} \cdot \mathbf{b} = \sum_{i=1}^{k} a_i \, b_i = \|\mathbf{a}\| \, \|\mathbf{b}\| \cos\theta$$

where $\theta$ is the angle between the two vectors. This single number tells you about **alignment**:

| Value of $\mathbf{a} \cdot \mathbf{b}$ | Geometric meaning |
|:---|:---|
| Large positive | $\mathbf{a}$ and $\mathbf{b}$ point in roughly the same direction |
| Zero | $\mathbf{a}$ and $\mathbf{b}$ are **perpendicular** (orthogonal) |
| Large negative | $\mathbf{a}$ and $\mathbf{b}$ point in roughly opposite directions |

The output is always a **scalar** — a single number, a grade-0 quantity. It answers "how much does $\mathbf{b}$ point in the direction of $\mathbf{a}$?" but tells you nothing about *which plane* the two vectors span or *what direction* the mismatch lies in. Keep this limitation in mind; we will return to it at the end of the chapter.

In [ ]:
# --- Dot product with simple vectors in R^4 ---

# Two vectors in R^4
a = np.array([1.0, 0.0, 0.0, 0.0])
b = np.array([0.6, 0.8, 0.0, 0.0])

dot_ab = np.dot(a, b)
cos_theta = dot_ab / (np.linalg.norm(a) * np.linalg.norm(b))
theta_deg = np.degrees(np.arccos(np.clip(cos_theta, -1, 1)))

print("=== Two generic vectors ===")
print(f"a = {a}")
print(f"b = {b}")
print(f"a . b = {dot_ab:.4f}")
print(f"cos(theta) = {cos_theta:.4f}")
print(f"theta = {theta_deg:.1f} degrees")
print()

# Parallel case: b is a scaled version of a
b_parallel = 3.0 * a
dot_parallel = np.dot(a, b_parallel)
cos_parallel = dot_parallel / (np.linalg.norm(a) * np.linalg.norm(b_parallel))
print("=== Parallel vectors ===")
print(f"a = {a}")
print(f"b = {b_parallel}")
print(f"a . b = {dot_parallel:.4f}")
print(f"cos(theta) = {cos_parallel:.4f}  (perfect alignment)")
print()

# Perpendicular case: b has zero overlap with a
b_perp = np.array([0.0, 1.0, 0.0, 0.0])
dot_perp = np.dot(a, b_perp)
print("=== Perpendicular vectors ===")
print(f"a = {a}")
print(f"b = {b_perp}")
print(f"a . b = {dot_perp:.4f}  (zero — no alignment at all)")

## Our Laboratory: Transformer Hidden States

When a transformer processes a prompt with $T$ tokens, it produces a hidden-state vector at every **(layer, token)** position. We can arrange these into a **layer-time grid**:

$$H^{(l,t)} \in \mathbb{R}^p, \quad l = 0, \ldots, L, \quad t = 1, \ldots, T.$$

- The **layer axis** ($l$) runs from the embedding layer ($l=0$) through each transformer block up to the final layer ($l=L$).
- The **time axis** ($t$) runs across the token positions in the input sequence.

Think of this grid as a two-dimensional field: at each grid point sits a high-dimensional vector. The `ltg_ga` library extracts this grid, whitens it (projects to a lower-dimensional isotropic subspace), and stores the result as `H_whitened` with shape $(L, T, k)$, where $k$ is the whitened dimension.

Let us load a model and run the analysis to see this grid in action.

In [ ]:
# Load a model and run the GA analysis
model = ltg_ga.load_model("Qwen/Qwen2.5-7B")
print(f"Model: {model.name}")
print(f"Layers: {model.n_layers}, Hidden dim: {model.hidden_dim}")

# Analyse a prompt
prompt = "The capital of France is Paris and the capital of Japan is"
result = ltg_ga.analyse(prompt, model=model, compute_dependency=False)

# Inspect the layer-time grid
H = result.H_whitened
L, T, k = H.shape
print(f"\nLayer-time grid shape: H_whitened.shape = {H.shape}")
print(f"  L = {L} layers (after skipping layer 0)")
print(f"  T = {T} tokens")
print(f"  k = {k} whitened dimensions")
print(f"\nTokens: {result.tokens}")

## Measuring Alignment in Hidden-State Space

Now that we have the hidden-state grid, we can ask: **how similar are two token representations at the same layer?**

The natural measure is **cosine similarity**, which normalizes away the vector lengths and focuses purely on direction:

$$\text{cos-sim}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \, \|\mathbf{v}\|}$$

A value of $+1$ means the vectors point in exactly the same direction; $0$ means they are orthogonal; $-1$ means they are antiparallel.

Let us compute the cosine similarity between every pair of tokens at a single layer and visualize it as a heatmap. This gives us a snapshot of the **representational geometry** at that layer: which tokens the model treats as similar, and which it distinguishes.

In [ ]:
# Cosine similarity between all token pairs at a fixed layer
def cosine_similarity_matrix(H_layer):
    """Compute the T x T cosine similarity matrix for a single layer.
    
    Args:
        H_layer: (T, k) array of hidden states at one layer.
    
    Returns:
        (T, T) cosine similarity matrix.
    """
    norms = np.linalg.norm(H_layer, axis=1, keepdims=True)
    H_normed = H_layer / (norms + 1e-12)
    return H_normed @ H_normed.T

# Pick a middle layer
mid_layer = L // 2
cos_sim = cosine_similarity_matrix(H[mid_layer])

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cos_sim, cmap='RdBu_r', vmin=-1, vmax=1, origin='lower')
ax.set_xlabel('Token position')
ax.set_ylabel('Token position')
ax.set_title(f'Cosine Similarity Between Tokens — Layer {mid_layer}')

# Add token labels if not too many
if T <= 15:
    ax.set_xticks(range(T))
    ax.set_xticklabels(result.tokens, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(T))
    ax.set_yticklabels(result.tokens, fontsize=8)

plt.colorbar(im, ax=ax, label='Cosine similarity')
fig.tight_layout()
plt.show()

print(f"Diagonal values (self-similarity): all {cos_sim[0,0]:.4f}")
print(f"Off-diagonal range: [{cos_sim[np.triu_indices(T, k=1)].min():.4f}, "
      f"{cos_sim[np.triu_indices(T, k=1)].max():.4f}]")

## How Alignment Changes Across Layers

One of the most striking geometric phenomena in transformers is that token representations tend to become **more similar** in the later layers. Early layers maintain distinct representations for each token, while late layers compress them toward a shared subspace (sometimes called the "residual stream convergence").

We can see this by comparing the cosine similarity heatmaps at an **early layer** (e.g., layer 2) and a **late layer** (e.g., layer 26). If the late-layer heatmap is more uniformly bright (high cosine similarity everywhere), it means the model has moved the token vectors closer together in direction — they are becoming more aligned.

This alignment is geometrically meaningful: it reflects the model converging on a shared representation for the output prediction, regardless of which token position we examine.

In [ ]:
# Compare cosine similarity at early vs late layers
early_layer = min(2, L - 1)
late_layer = min(26, L - 1)

cos_early = cosine_similarity_matrix(H[early_layer])
cos_late = cosine_similarity_matrix(H[late_layer])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Early layer
im1 = ax1.imshow(cos_early, cmap='RdBu_r', vmin=-1, vmax=1, origin='lower')
ax1.set_title(f'Early Layer {early_layer}')
ax1.set_xlabel('Token position')
ax1.set_ylabel('Token position')
if T <= 15:
    ax1.set_xticks(range(T))
    ax1.set_xticklabels(result.tokens, rotation=45, ha='right', fontsize=7)
    ax1.set_yticks(range(T))
    ax1.set_yticklabels(result.tokens, fontsize=7)
plt.colorbar(im1, ax=ax1, label='Cosine similarity')

# Late layer
im2 = ax2.imshow(cos_late, cmap='RdBu_r', vmin=-1, vmax=1, origin='lower')
ax2.set_title(f'Late Layer {late_layer}')
ax2.set_xlabel('Token position')
ax2.set_ylabel('Token position')
if T <= 15:
    ax2.set_xticks(range(T))
    ax2.set_xticklabels(result.tokens, rotation=45, ha='right', fontsize=7)
    ax2.set_yticks(range(T))
    ax2.set_yticklabels(result.tokens, fontsize=7)
plt.colorbar(im2, ax=ax2, label='Cosine similarity')

fig.suptitle('Token Alignment: Early vs Late Layers', fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig('ch01_cosine_layers.png', bbox_inches='tight', dpi=150)
print("Saved: ch01_cosine_layers.png")
plt.show()

# Quantify the difference
off_diag_early = cos_early[np.triu_indices(T, k=1)]
off_diag_late = cos_late[np.triu_indices(T, k=1)]
print(f"\nMean off-diagonal cosine similarity:")
print(f"  Layer {early_layer} (early): {off_diag_early.mean():.4f}")
print(f"  Layer {late_layer} (late):  {off_diag_late.mean():.4f}")
print(f"  Change: {off_diag_late.mean() - off_diag_early.mean():+.4f}")

## What the Dot Product Misses

The inner product is a powerful tool — it tells us **how much** two vectors are aligned. But notice what it does *not* tell us:

1. **Which plane do the vectors span?** Two vectors in $\mathbb{R}^{256}$ that make a 45-degree angle span a specific 2D plane out of the $\binom{256}{2} = 32{,}640$ possible planes. The dot product gives the same scalar $\cos(45°)$ regardless of *which* plane.

2. **What is the oriented area?** If $\mathbf{a}$ and $\mathbf{b}$ are not parallel, the parallelogram they span has an area $\|\mathbf{a}\| \, \|\mathbf{b}\| \sin\theta$. The dot product involves $\cos\theta$; the area involves $\sin\theta$. The dot product discards the $\sin\theta$ part entirely.

3. **No direction of mismatch.** Knowing that two token representations have cosine similarity 0.85 does not tell you *in which directions* they differ. The full geometric relationship requires knowing not just the scalar overlap but also the **plane of rotation** that would bring one vector toward the other.

In other words, the dot product is a **lossy compression** of the geometric relationship between two vectors. It reduces the full relationship to a single number.

In Chapter 2, we will meet the **geometric product** $\mathbf{a}\mathbf{b} = \mathbf{a} \cdot \mathbf{b} + \mathbf{a} \wedge \mathbf{b}$, which captures *both* the scalar alignment ($\cos\theta$) and the oriented plane ($\sin\theta$ plus the plane directions) in a single algebraic object. The dot product is the grade-0 part; the wedge product $\mathbf{a} \wedge \mathbf{b}$ is the grade-2 part (a **bivector**) that encodes everything the dot product loses.

## Exercises

**Exercise 1.1 — Projection magnitude.**
Pick two tokens from the prompt (e.g., "France" and "Japan") and extract their hidden-state vectors at layers 5, 15, and 25. Compute the **projection** of one onto the other at each layer: $\text{proj} = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{b}\|}$. Does the projection magnitude increase, decrease, or stay constant as you go deeper?

**Exercise 1.2 — Cosine similarity trajectory.**
For a single pair of tokens, compute the cosine similarity at *every* layer (from layer 0 to layer $L-1$) and plot it as a curve. At which layer does the similarity change most rapidly? Compare this to the rotor angle profile from `result.angles` — is there a connection?

**Exercise 1.3 — The information the dot product discards.**
Take two token vectors $\mathbf{a}$ and $\mathbf{b}$ at a layer where their cosine similarity is approximately 0.5 (i.e., $\theta \approx 60°$). Compute $\|\mathbf{a}\| \, \|\mathbf{b}\| \sin\theta$ — this is the area of the parallelogram they span, and it measures the magnitude of the information the dot product throws away. How does this "discarded area" compare in magnitude to the dot product itself? What does this suggest about how much geometric structure is invisible to cosine similarity?